# CIFAR-10 Classification with Pure CNN

This notebook implements a pure CNN (no pretrained models) for classifying CIFAR-10 dataset.

**Key Features:**
- Pure CNN architecture (no ResNet, VGG, etc.)
- Data augmentation to prevent overfitting
- Early stopping and learning rate scheduling
- Data balancing with weighted sampling
- Target: >90% validation accuracy

In [ ]:
# CIFAR-10 with pure CNN

import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10

In [ ]:
# -----------------------------
# 1) Reproducibility and device
# -----------------------------

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# AMP for GPU
USE_AMP = device.type == "cuda"
if USE_AMP:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler('cuda')
    print("Mixed precision (AMP) enabled")

In [ ]:
# -----------------------------
# 2) Load CIFAR-10 dataset
# -----------------------------

print("Loading CIFAR-10 dataset...")

IMG_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 4 if device.type == "cuda" else 0

mean = [0.4914, 0.4822, 0.4465]
std = [0.2470, 0.2435, 0.2616]

train_tfms = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

val_tfms = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

train_dataset = CIFAR10(root="./data", train=True, download=True, transform=train_tfms)
val_dataset = CIFAR10(root="./data", train=False, download=True, transform=val_tfms)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")

In [ ]:
# -----------------------------
# 3) Data analysis
# -----------------------------

train_labels = [sample[1] for sample in train_dataset]
val_labels = [sample[1] for sample in val_dataset]

train_counts = np.bincount(train_labels, minlength=10)
val_counts = np.bincount(val_labels, minlength=10)

print("\nClass distribution (train):")
for i, (name, count) in enumerate(zip(train_dataset.classes, train_counts)):
    print(f"  {i}: {name} - {count}")

print("\nClass distribution (val):")
for i, (name, count) in enumerate(zip(val_dataset.classes, val_counts)):
    print(f"  {i}: {name} - {count}")

# Plot
plt.figure(figsize=(12, 4))
x = np.arange(10)
w = 0.35
plt.bar(x - w/2, train_counts, width=w, label="train")
plt.bar(x + w/2, val_counts, width=w, label="val")
plt.xticks(x, [c[:3] for c in train_dataset.classes], rotation=45)
plt.ylabel("Count")
plt.title("Class distribution")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# 4) Data balancing
# -----------------------------

train_labels_array = np.array(train_labels)
class_sample_count = np.bincount(train_labels_array, minlength=10)
class_weights = 1.0 / np.maximum(class_sample_count, 1)
sample_weights = class_weights[train_labels_array]
sample_weights = torch.from_numpy(sample_weights).double()

sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print("Data loaders prepared with weighted sampling for balancing")

In [ ]:
# -----------------------------
# 5) Pure CNN model
# -----------------------------

class CIFAR10CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 32x32 -> 16x16
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
            
            # Block 2: 16x16 -> 8x8
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),
            
            # Block 3: 8x8 -> 4x4
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.4),
            
            # Block 4: 4x4 -> 2x2
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = CIFAR10CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-6
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# -----------------------------
# 6) Training loop
# -----------------------------

EPOCHS = 50
PATIENCE = 10
best_val_acc = 0.0
best_state = None
no_improve = 0

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}


def run_one_epoch(loader, train_mode=True):
    if train_mode:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            if USE_AMP and train_mode:
                with autocast('cuda'):
                    logits = model(images)
                    loss = criterion(logits, labels)
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(images)
                loss = criterion(logits, labels)
                if train_mode:
                    optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
                    optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / max(total, 1)
    epoch_acc = correct / max(total, 1)
    return epoch_loss, epoch_acc


for epoch in range(EPOCHS):
    train_loss, train_acc = run_one_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_one_epoch(val_loader, train_mode=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    scheduler.step(val_acc)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

if best_state is not None:
    model.load_state_dict(best_state)

In [ ]:
# -----------------------------
# 7) Evaluation
# -----------------------------

val_loss, val_acc = run_one_epoch(val_loader, train_mode=False)
print(f"\nBest/Final validation accuracy: {val_acc*100:.2f}%")

# Confusion matrix
from sklearn.metrics import confusion_matrix

all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm)

precision = np.diag(cm) / np.maximum(cm.sum(axis=0), 1)
recall = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-8)

print("\nPer-class metrics:")
for i, name in enumerate(train_dataset.classes):
    print(f"  {name}: P={precision[i]:.4f}, R={recall[i]:.4f}, F1={f1[i]:.4f}")

In [ ]:
# -----------------------------
# 8) Plot training history
# -----------------------------

epochs_ran = len(history["train_loss"])
x = np.arange(1, epochs_ran + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(x, history["train_loss"], label="train loss")
plt.plot(x, history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss curve")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)

plt.subplot(1, 2, 2)
plt.plot(x, np.array(history["train_acc"]) * 100, label="train acc")
plt.plot(x, np.array(history["val_acc"]) * 100, label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy curve")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# 9) Save model
# -----------------------------

save_path = "cifar10_pure_cnn_best.pth"
torch.save(model.state_dict(), save_path)
print(f"Saved best model to: {save_path}")

if val_acc >= 0.90:
    print("Target reached: validation accuracy >= 90%")
else:
    print("Validation accuracy < 90%. Consider tuning hyperparameters.")

## Summary

This notebook demonstrates:
1. **Pure CNN Architecture**: Custom CNN with no pretrained models
2. **CIFAR-10 Dataset**: 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck)
3. **Anti-overfitting Techniques**:
   - Data augmentation (RandomHorizontalFlip, RandomRotation, ColorJitter, RandomAffine)
   - Dropout (0.5, 0.4, 0.3, 0.2)
   - Batch normalization
   - Weight decay (L2 regularization)
   - Early stopping
   - Learning rate scheduling
   - Gradient clipping
4. **Data Balancing**: WeightedRandomSampler
5. **Target**: Achieve >90% validation accuracy